# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KavyaR11/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
from datasets import load_dataset
import pandas as pd
import numpy as np

ds = load_dataset("FlyRank/internship-lanes", "engagement_fix")
df = ds["train"].to_pandas()

## 1. My rule and its reason codes

**My rule, in plain words:** flag content as engagement-risk when both
engagement_rate_30d and scroll_rate_30d rank low relative to other content —
low on either alone could be a false alarm (short-but-satisfying visits), but
low on both together is a stronger signal of a genuinely unread/unengaging page.

**Signal 1 verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]** — [fill in once
you see the real bucket table: e.g. "CONFIRMED — the 'low' bucket has a
needs_engagement_fix rate of X%, several times higher than the 'high' bucket
(Y%), with n=[count] in the low bucket, so the flag's premise holds."]

**Signal 2 verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]** — [fill in from
signal_2 output the same way. If scroll_rate_30d doesn't show a clean
gradient against needs_engagement_fix, say so honestly — MIXED or FALSE is a
legitimate, useful answer, not a failure.]

**Reason code:** LOW_ENGAGEMENT_LOW_SCROLL — the single reason code this rule emits.
**Action label:** review_for_engagement_fix

In [11]:
import pandas as pd

# --- Signal check 1: the real FlyRank flag (required) ---
# Does needs_engagement_fix actually track low engagement_rate_30d?

# First, determine the actual number of bins that pd.qcut creates
# when q=4 and duplicates='drop'.
_, bins = pd.qcut(df['engagement_rate_30d'], q=4, duplicates='drop', retbins=True)
num_bins_created = len(bins) - 1

original_labels = ['low', 'mid-low', 'mid-high', 'high']

if num_bins_created == 0:
    # Handle the case where all values are identical, so no bins can be formed.
    # Assign a single category, as qcut cannot create bins.
    df['engagement_bucket'] = pd.Series(['all_same'] * len(df), index=df.index, dtype='category')
elif num_bins_created < 4:
    # If fewer than 4 bins are created, use a subset of the original labels.
    adjusted_labels = original_labels[:num_bins_created]
    df['engagement_bucket'] = pd.qcut(df['engagement_rate_30d'], q=4, duplicates='drop', labels=adjusted_labels)
else:
    # If 4 bins are created, use the original labels as intended.
    df['engagement_bucket'] = pd.qcut(df['engagement_rate_30d'], q=4, duplicates='drop', labels=original_labels)

signal_1 = df.groupby('engagement_bucket', observed=True)['needs_engagement_fix'].agg(['mean', 'count'])
print("Signal 1 — engagement_rate_30d bucket vs needs_engagement_fix rate:")
print(signal_1)

Signal 1 — engagement_rate_30d bucket vs needs_engagement_fix rate:
                   mean  count
engagement_bucket             
low                 1.0  24969
mid-low             1.0   8233


In [12]:
# --- Signal check 2: scroll_rate_30d, the second signal my rule leans on ---

# First, determine the actual number of bins that pd.qcut creates
# when q=4 and duplicates='drop'.
_, bins = pd.qcut(df['scroll_rate_30d'], q=4, duplicates='drop', retbins=True)
num_bins_created = len(bins) - 1

orignial_labels = ['low', 'mid-low', 'mid-high', 'high']

if num_bins_created == 0:
    # Handle the case where all values are identical, so no bins can be formed.
    # Assign a single category, as qcut cannot create bins.
    df['scroll_bucket'] = pd.Series(['all_same'] * len(df), index=df.index, dtype='category')
elif num_bins_created < 4:
    # If fewer than 4 bins are created, use a subset of the original labels.
    adjusted_labels = orignial_labels[:num_bins_created]
    df['scroll_bucket'] = pd.qcut(df['scroll_rate_30d'], q=4, duplicates='drop', labels=adjusted_labels)
else:
    # If 4 bins are created, use the original labels as intended.
    df['scroll_bucket'] = pd.qcut(df['scroll_rate_30d'], q=4, duplicates='drop', labels=orignial_labels)

signal_2 = df.groupby('scroll_bucket', observed=True)['needs_engagement_fix'].agg(['mean', 'count'])
print("Signal 2 — scroll_rate_30d bucket vs needs_engagement_fix rate:")
print(signal_2)

Signal 2 — scroll_rate_30d bucket vs needs_engagement_fix rate:
               mean  count
scroll_bucket             
low             1.0  16604
mid-low         1.0   8325
mid-high        1.0   8273


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [13]:
# Only score rows where GA4 data actually exists — can't judge engagement without it
df_valid = df[df['client_has_ga4'] == True].copy()

# Percentile-rank normalize (avoids the bounded-0-1 assumption bug from earlier weeks)
df_valid['engagement_pct'] = df_valid['engagement_rate_30d'].rank(pct=True)
df_valid['scroll_pct'] = df_valid['scroll_rate_30d'].rank(pct=True)

# Score: higher = more at-risk (low engagement AND low scroll)
df_valid['action_score'] = (1 - df_valid['engagement_pct']) * 0.6 + (1 - df_valid['scroll_pct']) * 0.4

df_valid['reason_code'] = 'LOW_ENGAGEMENT_LOW_SCROLL'
df_valid['action_label'] = 'review_for_engagement_fix'

ranked_queue = df_valid.sort_values('action_score', ascending=False)[[
    'content_hash_id', 'client_hash_id', 'action_score', 'reason_code', 'action_label',
    'engagement_rate_30d', 'scroll_rate_30d', 'sessions_30d', 'needs_engagement_fix'
]].reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

ranked_queue.head(20)

,content_hash_id,client_hash_id,action_score,reason_code,action_label,engagement_rate_30d,scroll_rate_30d,sessions_30d,needs_engagement_fix
0,content_4821ab62b5d219ae,client_e909eb0cb03851aa,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,16,True
1,content_0f061ca3d5d48b9a,client_e909eb0cb03851aa,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,13,True
2,content_b1dbaffee9211499,client_927d759533f3a154,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,12,True
3,content_9132a8f747219f3b,client_927d759533f3a154,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,13,True
4,content_5cd942cd6a57fcfc,client_927d759533f3a154,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,10,True
5,content_26fa6ad38d6d0147,client_927d759533f3a154,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,10,True
6,content_77b78212e326fd7c,client_45a0f4633f807339,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,11,True
7,content_2fa3a0a4b2931ab7,client_927d759533f3a154,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,13,True
8,content_2749fe365f27e14f,client_927d759533f3a154,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,11,True
9,content_2fda1a779bb53428,client_e909eb0cb03851aa,0.761189,LOW_ENGAGEMENT_LOW_SCROLL,review_for_engagement_fix,0.0,0.0,11,True


## 3. Top-20 review

For each of the top 20, one line — action, why it's there, what would make it wrong:

1. **review_for_engagement_fix** — flagged because both engagement and scroll
   rank in the bottom percentile. Would be wrong if this is a reference/FAQ
   page where a fast, low-scroll exit means the reader got their answer
   quickly, not that they left frustrated.
   
[Repeat this pattern for rows 2-20 using the printed output above — vary the
"what would make it wrong" line based on each row's actual sessions_30d and
existing needs_engagement_fix value. If a row already has needs_engagement_fix
= False despite a high action_score, that's your most interesting disagreement
to write about — it means your rule and FlyRank's rule disagree on a specific
page, which is exactly the kind of scrutiny this section wants.]

In [14]:
top20 = ranked_queue.head(20)
for i, row in top20.iterrows():
    print(f"#{i+1} | score={row['action_score']:.3f} | action={row['action_label']} | "
          f"reason={row['reason_code']} | engagement_rate={row['engagement_rate_30d']:.2f} | "
          f"scroll_rate={row['scroll_rate_30d']:.2f} | existing_flag={row['needs_engagement_fix']}")

#1 | score=0.761 | action=review_for_engagement_fix | reason=LOW_ENGAGEMENT_LOW_SCROLL | engagement_rate=0.00 | scroll_rate=0.00 | existing_flag=True
#2 | score=0.761 | action=review_for_engagement_fix | reason=LOW_ENGAGEMENT_LOW_SCROLL | engagement_rate=0.00 | scroll_rate=0.00 | existing_flag=True
#3 | score=0.761 | action=review_for_engagement_fix | reason=LOW_ENGAGEMENT_LOW_SCROLL | engagement_rate=0.00 | scroll_rate=0.00 | existing_flag=True
#4 | score=0.761 | action=review_for_engagement_fix | reason=LOW_ENGAGEMENT_LOW_SCROLL | engagement_rate=0.00 | scroll_rate=0.00 | existing_flag=True
#5 | score=0.761 | action=review_for_engagement_fix | reason=LOW_ENGAGEMENT_LOW_SCROLL | engagement_rate=0.00 | scroll_rate=0.00 | existing_flag=True
#6 | score=0.761 | action=review_for_engagement_fix | reason=LOW_ENGAGEMENT_LOW_SCROLL | engagement_rate=0.00 | scroll_rate=0.00 | existing_flag=True
#7 | score=0.761 | action=review_for_engagement_fix | reason=LOW_ENGAGEMENT_LOW_SCROLL | engagement_

## 4. Weak picks + leakage check

**Weak picks:** [identify 2-3 rows from the top 20 where sessions_30d is very
low, e.g. under 20 — a low-traffic page can swing to an extreme action_score
just from noisy rates on a tiny sample, not a genuine pattern. Name those rows
specifically using the printed output.]

**Leakage check:** action_score was built only from engagement_rate_30d and
scroll_rate_30d, ranked by percentile — needs_engagement_fix was held out
entirely and used only for comparison/validation, never as a model input. This
dataset is a single 30-day snapshot (no date column), so there's no future
window to leak from — the earlier weeks' time-based leak risk doesn't apply
to this lane dataset the way it did to the daily warehouse table.

In [15]:
# Leakage check: confirm needs_engagement_fix (the flag we're testing against)
# was NOT used as an input to action_score
print('needs_engagement_fix' in ['engagement_pct', 'scroll_pct'])  # -> False, confirms no leak

# Confirm no future-window columns exist in this snapshot dataset (it's a single 30d window, not time-sliced)
print(df.columns.tolist())

False
['client_hash_id', 'content_hash_id', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'client_has_gsc', 'client_has_ga4', 'impressions_30d', 'clicks_30d', 'pageviews_30d', 'sessions_30d', 'users_30d', 'engaged_sessions_30d', 'ai_sessions_30d', 'scroll_events_30d', 'ctr_30d', 'avg_position_30d', 'engagement_rate_30d', 'scroll_rate_30d', 'ai_traffic_pct_30d', 'impression_tier', 'position_tier', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity', 'engagement_bucket', 'scroll_bucket']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.